# 🧪 W8-D2 概念实验：ApplicationContract

> 配套阅读：`第8周-Day2-ApplicationContract.md`（ADR-005、Gap Analysis 在那边）
>
> md 讲了 Contract 的"为什么"；这里把四个核心机制写成**可执行代码**：
> 1. **两层分离**：ApplicationContract（稳定标识+生命周期） vs Version（不可变内容快照）
> 2. **内容寻址**：对规范化内容算 sha256 指纹——"同内容必同指纹、改一字必变指纹"
> 3. **传输无关**：同一 Version 绑定 HTTP / OpenAI tool / CLI 三种传输，语义指纹不变
> 4. **破坏性变更分类器**：把 ADR-005 §4.2 的规则写成可机械判定的 diff 函数

环境：仅标准库 + numpy/matplotlib（无 pydantic、无网络）。

## 实验 1：两层模型 + 内容寻址指纹

- `Contract`：稳定业务标识（类比 ISBN），管生命周期 Draft→Stable→Deprecated→Retired
- `ContractVersion`：不可变内容快照（类比版次），承载输入输出 Schema、权限/策略边界

铁律 HC-10：**Version 一经发布不可修改**——用 `frozen dataclass` 强制。
指纹 = sha256(规范化 JSON)：key 排序 + 紧凑分隔符，保证"内容等价 ⇒ 指纹等价"。

In [ ]:
import hashlib, json
from dataclasses import dataclass, field, replace

def fingerprint(payload: dict) -> str:
    """内容寻址指纹：规范化 JSON（排序+紧凑分隔）→ sha256 前 16 位。"""
    canonical = json.dumps(payload, ensure_ascii=False, sort_keys=True,
                           separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:16]

@dataclass(frozen=True)
class ContractVersion:
    contract_id: str
    semver: str
    payload: dict = field(compare=False)      # 内容不进 equality，指纹才是身份证
    fp: str = ""

    def __post_init__(self):
        object.__setattr__(self, "fp", fingerprint(self.payload))

@dataclass
class Contract:
    contract_id: str
    lifecycle: str = "Draft"                  # Draft→Stable→Deprecated→Retired
    versions: list = None
    def __post_init__(self): self.versions = []
    def publish(self, version: ContractVersion):
        if version.contract_id != self.contract_id:
            raise ValueError("version 挂错了 contract")
        self.versions.append(version)
        self.lifecycle = "Stable"

v10 = ContractVersion("contract.ticket.query", "1.0.0", {
    "input_schema":  {"ticket_id": "string", "required": ["ticket_id"]},
    "output_schema": {"status": "string", "eta": "string"},
    "required_scopes": ["ticket:read"],
    "effect_policy": "read_only",
})
c = Contract("contract.ticket.query")
c.publish(v10)

# --- 铁律1：不可变 ---
try:
    v10.effect_policy = "conditional_write"          # type: ignore
except Exception as e:
    print(f"✓ 试图修改已发布 Version → 被拒绝：{type(e).__name__}")

# --- 铁律2：内容等价 ⇒ 指纹等价（key 顺序无关）---
same = {"effect_policy": "read_only", "required_scopes": ["ticket:read"],
        "output_schema": {"eta": "string", "status": "string"},
        "input_schema": {"required": ["ticket_id"], "ticket_id": "string"}}
print(f"✓ key 顺序不同、内容相同 → 指纹一致：{fingerprint(same) == v10.fp}")

# --- 铁律3：改一字必变指纹 ---
payload_11 = {**v10.payload, "output_schema": {**v10.payload["output_schema"], "priority": "string"}}
v11 = ContractVersion("contract.ticket.query", "1.1.0", payload_11)   # 新增可选输出字段
c.publish(v11)
print(f"✓ 新增字段后指纹变化：{v10.fp} → {v11.fp}")
print(f"Contract '{c.contract_id}' 生命周期={c.lifecycle}，共存版本：",
      [(v.semver, v.fp) for v in c.versions])

## 实验 2：传输无关原则（ADR-005 §4.1）

同一份 ContractVersion 绑定到 **HTTP JSON / OpenAI tool schema / CLI argv** 三种传输，
各自序列化完全不同——但反解析回语义载荷后，**指纹与原指纹完全一致**。
这就是"Contract 不变，只有适配层变"的可执行版。

In [ ]:
import shlex

v = c.versions[0]   # 拿 1.0.0 版

# --- 传输绑定层：三种 adapter，各自面向一种 wire ---
def to_http(v):     # HTTP + JSON body
    return {"url": f"/api/v1/contracts/{v.contract_id}", "method": "POST",
            "body": v.payload}

def to_openai_tool(v):   # OpenAI function calling 的 tool schema
    return {"type": "function", "function": {
        "name": v.contract_id.replace(".", "_"),
        "description": "查询工单状态（来自 ApplicationContract）",
        "parameters": v.payload["input_schema"]}}

def to_cli(v):      # CLI argv
    return ("langchat contract run " + v.contract_id +
            " --input " + shlex.quote(json.dumps({"ticket_id": "<id>"}, ensure_ascii=False)))

wire_http, wire_tool, wire_cli = to_http(v), to_openai_tool(v), to_cli(v)
print("HTTP 绑定:", json.dumps(wire_http, ensure_ascii=False)[:90], "...")
print("OpenAI 绑定:", json.dumps(wire_tool, ensure_ascii=False)[:90], "...")
print("CLI  绑定:", wire_cli[:90], "...")

# --- 各自反解析回语义载荷，重算指纹 ---
back_http = json.loads(json.dumps(wire_http["body"]))                 # 从 HTTP body 还原
back_tool = {"input_schema": wire_tool["function"]["parameters"]}     # 从 tool schema 还原(部分视图)
roundtrip_http = fingerprint(back_http)
roundtrip_tool = fingerprint(back_tool)
print(f"\n原指纹                : {v.fp}")
print(f"HTTP 往返指纹          : {roundtrip_http}  一致={roundtrip_http == v.fp}")
print(f"OpenAI 往返(部分视图)  : {roundtrip_tool}  一致={roundtrip_tool == v.fp}")
print("\n注意第三行为 False：tool schema 只承载了 input_schema 这一个视图，")
print("指纹不等恰恰证明——传输层裁剪会造成语义丢失，完整指纹必须锚定在 Version 全量内容上。")
print("=> 换传输协议不需要动 Contract；但任何'视图/裁剪'都不再是同一份契约。")

## 实验 3：破坏性变更分类器（ADR-005 §4.2 的可执行版）

md 里的 13 条规则，挑最核心的 6 条写成 `classify_change(old, new)`：
删字段/收紧输入/改语义/收紧策略 → **MAJOR**；新增可选/放宽输出 → **MINOR**。
再写一个 `guard`：MINOR 只许递增次版本号，MAJOR 必须递增主版本号——CI 里就能挡住"悄悄破坏"。

In [ ]:
def classify_change(old: dict, new: dict) -> list:
    """返回变更清单 [(规则, 分类)]。"""
    changes = []
    old_in, new_in = old["input_schema"], new["input_schema"]
    old_out, new_out = old["output_schema"], new["output_schema"]
    for f in set(old_in) - set(new_in):
        changes.append((f"删除输入字段 {f}", "MAJOR"))
    req_old = set(old_in.get("required", []))
    req_new = set(new_in.get("required", []))
    if req_new - req_old:
        changes.append((f"收紧输入(新增必填 {req_new - req_old})", "MAJOR"))
    for f in set(new_in) - set(old_in):
        if f != "required" and f not in req_new:
            changes.append((f"新增可选输入字段 {f}", "MINOR"))
    for f in set(old_out) - set(new_out):
        changes.append((f"删除输出字段 {f}", "MAJOR"))
    for f in set(new_out) - set(old_out):
        changes.append((f"放宽输出(新增输出字段 {f})", "MINOR"))
    if old["effect_policy"] != new["effect_policy"]:
        changes.append((f"effect_policy: {old['effect_policy']}→{new['effect_policy']}", "MAJOR"))
    if set(new["required_scopes"]) - set(old["required_scopes"]):
        changes.append((f"required_scopes 新增必需 scope", "MAJOR"))
    return changes or [("无实质变更", "PATCH")]

def semver_guard(changes, old_ver, new_ver):
    """版本号闸门：有 MAJOR 必须升主版本；只有 MINOR/PATCH 不得升主版本。"""
    levels = {c[1] for c in changes}
    old_major, new_major = int(old_ver[0]), int(new_ver[0])
    if "MAJOR" in levels and new_major <= old_major:
        return f"✗ 拒绝：含破坏性变更但版本号 {old_ver}→{new_ver} 没升主版本"
    if "MAJOR" not in levels and new_major > old_major:
        return f"✗ 拒绝：无破坏性变更却升了主版本 {old_ver}→{new_ver}"
    return f"✓ 放行 {old_ver}→{new_ver}"

base = v10.payload
cases = [
    ("新增可选输出字段", {**base, "output_schema": {**base["output_schema"], "priority": "string"}}, "1.1.0"),
    ("收紧输入(新增必填)→应拒", {**base, "input_schema": {**base["input_schema"],
        "lang": "string", "required": ["ticket_id", "lang"]}}, "1.2.0"),
    ("删除输出字段(升主版本)", {**base, "output_schema": {"status": "string"}}, "2.0.0"),
    ("策略升级 read_only→写", {**base, "effect_policy": "conditional_write"}, "2.0.0"),
    ("悄悄破坏(删字段却不升主版本)", {**base, "output_schema": {"status": "string"}}, "1.4.0"),
]
for name, new_payload, new_ver in cases:
    changes = classify_change(base, new_payload)
    verdict = semver_guard(changes, v10.semver, new_ver)
    detail = "; ".join(f"{r}[{lv}]" for r, lv in changes)
    print(f"{name}:\n  变更: {detail}\n  闸门: {verdict}\n")

## 实验 4：可视化 —— 为什么 API 文档替代不了 Contract

In [ ]:
# matplotlib 中文字体配置（NotoSansCJK，每次画图前先跑这段）
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = fontManager_font = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

dims = ["机器可校验", "内容寻址指纹", "破坏性变更判定", "版本共存", "CI可挡违规", "跨传输复用"]
api_doc = [1, 0, 0, 1, 0, 1]        # 文档：人读的，机器只能'看'
contract = [5, 5, 5, 5, 5, 5]       # 代码化契约：本 notebook 实验1-3 都演示过

x = np.arange(len(dims))
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(x - 0.2, api_doc, 0.4, color="#adb5bd", label="API 文档（散文）")
ax.barh(x + 0.2, contract, 0.4, color="#fb8500", label="ApplicationContract（本实验）")
ax.set_yticks(x); ax.set_yticklabels(dims, fontsize=10)
ax.invert_yaxis(); ax.set_xlim(0, 5.6)
ax.set_title("API 文档 vs ApplicationContract：六项工程能力对比（实验1-3 均已演示右列）")
ax.legend(fontsize=10, loc="lower right")
for i, (a, b) in enumerate(zip(api_doc, contract)):
    ax.text(b + 0.1, i + 0.2, f"{b}/5", va="center", fontsize=9, color="#fb8500")
    ax.text(a + 0.1, i - 0.2, f"{a}/5", va="center", fontsize=9, color="#6c757d")
plt.tight_layout(); plt.show()

print("md 的 Gap Analysis 结论在这里变成可验证的断言：")
print("Contract 把'约束'从人读的散文变成机器可判定的结构——")
print("指纹锚定内容(实验1)、传输无关(实验2)、变更分类+版本闸门(实验3)。")

## 小结

- 两层分离：Contract=稳定标识+生命周期；Version=不可变内容快照（frozen 强制 HC-10）
- 内容寻址：规范化 JSON→sha256，内容等价⇔指纹等价；传输层裁剪≠同一契约
- 破坏性变更可机械判定，版本号闸门可以进 CI——"悄悄破坏"被结构性挡住
- 对应 md Gap：当前 SkillReleaseDescriptor 已承载部分职责，缺的是 Version 分层与指纹